In [7]:
# ============================================================
# Functional Dependency Discovery — Google Colab Runnable
# Run each cell in order. All dependencies are installed
# in the first cell.
# ============================================================

# ── CELL 1: Install dependencies ────────────────────────────
# !pip install plotly pandas numpy

# ── CELL 2: Imports ─────────────────────────────────────────
import json
import sys
import time
from itertools import combinations
from string import ascii_lowercase

import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go

# ── CELL 3: Build dataset ────────────────────────────────────
np.random.seed(42)

course_data = {
    'C01': ('Algorithms',     4, 'CS',   'Tech Hall'),
    'C02': ('Databases',      3, 'CS',   'Tech Hall'),
    'C03': ('Calculus',       4, 'Math', 'Science Bldg'),
    'C04': ('Statistics',     3, 'Math', 'Science Bldg'),
    'C05': ('Physics',        4, 'Phys', 'Science Bldg'),
    'C06': ('Linear Algebra', 3, 'Math', 'Science Bldg'),
    'C07': ('OS',             3, 'CS',   'Tech Hall'),
    'C08': ('Networks',       3, 'CS',   'Tech Hall'),
}
majors     = ['CS', 'Math', 'Physics', 'Engineering']
years      = ['Freshman', 'Sophomore', 'Junior', 'Senior']
grades     = ['A', 'B', 'C', 'D', 'F']
course_ids = list(course_data.keys())
student_ids = [f'S{i:03d}' for i in range(1, 201)]

rows = []
for sid in student_ids:
    major = np.random.choice(majors)
    year  = np.random.choice(years)
    for cid in np.random.choice(course_ids, np.random.randint(2, 6), replace=False):
        cname, credits, dept, bldg = course_data[cid]
        grade = np.random.choice(grades, p=[0.25, 0.35, 0.25, 0.1, 0.05])
        rows.append({
            'StudentID':  sid,
            'Major':      major,
            'Year':       year,
            'CourseID':   cid,
            'CourseName': cname,
            'Credits':    str(credits),
            'Department': dept,
            'Building':   bldg,
            'Grade':      grade,
        })

DF = pd.DataFrame(rows)
print(f"Shape: {DF.shape}")
print(DF.head())

# ── CELL 4: Core FD engine ───────────────────────────────────
def card_of_partition(candidate, df):
    if len(candidate) == 1:
        return df[candidate[0]].nunique()
    return df.drop_duplicates(list(candidate)).shape[0]


def find_fds(df, max_k=2):
    U = list(df.columns)
    fds_1, fds_2 = [], []
    for lhs in U:
        for rhs in U:
            if lhs == rhs:
                continue
            if (df.groupby(lhs)[rhs].nunique() == 1).all():
                fds_1.append(([lhs], rhs))
    for (a, b) in combinations(U, 2):
        for rhs in U:
            if rhs in [a, b]:
                continue
            if (df.groupby([a, b])[rhs].nunique() == 1).all():
                fds_2.append(([a, b], rhs))
    return fds_1, fds_2


def find_equivalences(fds_1):
    equivs, seen = [], set()
    fd_set = {(tuple(lhs), rhs) for lhs, rhs in fds_1}
    for lhs, rhs in fds_1:
        a = lhs[0]
        if (tuple([rhs]), a) in fd_set:
            pair = tuple(sorted([a, rhs]))
            if pair not in seen:
                seen.add(pair)
                equivs.append((a, rhs))
    return equivs


def find_keys(df):
    U, n, keys = list(df.columns), len(df), []
    for k in range(1, len(U) + 1):
        for subset in combinations(U, k):
            if df.drop_duplicates(list(subset)).shape[0] == n:
                if not any(set(key).issubset(set(subset)) for key in keys):
                    keys.append(list(subset))
    return keys

# ── CELL 5: Run discovery ────────────────────────────────────
start = time.time()

fds_1, fds_2   = find_fds(DF)
equivs          = find_equivalences(fds_1)
keys            = find_keys(DF)

colsName        = list(DF.columns)
dic             = {c: i for i, c in enumerate(colsName)}

Uni_FD, Uni_FD_set1, Uni_FD_set2, Uni_E, Uni_K = [], [], [], [], []

print("Functional Dependencies:")
for lhs, rhs in fds_1:
    s = "{" + ", ".join(lhs) + "} -> {" + rhs + "}"
    print(" ", s)
    if s not in Uni_FD:
        Uni_FD.append(s)
    Uni_FD_set1.append([lhs, rhs])

for lhs, rhs in fds_2:
    s = "{" + ", ".join(lhs) + "} -> {" + rhs + "}"
    if s not in Uni_FD:
        Uni_FD.append(s)
    Uni_FD_set2.append([lhs, rhs])

print("\nEquivalences:")
for a, b in equivs:
    s = "{" + a + "} <-> {" + b + "}"
    print(" ", s)
    if s not in Uni_E:
        Uni_E.append(s)

print("\nKeys:")
for key in keys:
    s = "{" + ", ".join(key) + "}"
    print(" ", s)
    if s not in Uni_K:
        Uni_K.append(s)

print(f"\nTime: {round(time.time() - start, 4)}s")
print(f"Summary: {len(fds_1)} 1->1 FDs | {len(fds_2)} 2->1 FDs | {len(equivs)} equivalences | {len(keys)} keys")

# ── CELL 6: Sankey diagram (1-on-1 FDs) ─────────────────────
opacity = 0

source = [dic[item[0][0]] for item in Uni_FD_set1]
target = [dic[item[1]]    for item in Uni_FD_set1]
value  = [1] * len(source)

colors_node = [list(np.random.choice(range(256), size=3)) for _ in colsName]

colors_link = [
    f"rgba({colors_node[s][0]},{colors_node[s][1]},{colors_node[s][2]},{opacity})"
    for s in source
]
colors_node_str = [f"rgb({c[0]},{c[1]},{c[2]})" for c in colors_node]

link = dict(source=source, target=target, value=value,
            color=colors_link, line=dict(width=0.1))
node = dict(label=colsName, pad=25, thickness=25,
            line=dict(color="black", width=0.5), color=colors_node_str)

fig_sankey = go.Figure(
    go.Sankey(link=link, node=node),
    go.Layout(
        title_text="1-on-1 Functional Dependencies — Sankey Diagram",
        autosize=False, width=800, height=600,
        font=dict(size=13)
    )
)
fig_sankey.show()

div1_txt = plotly.offline.plot(fig_sankey, include_plotlyjs=False, output_type="div")
with open("1on1_dependency_sankey.html", "w") as f:
    f.write("<script src='https://cdn.plot.ly/plotly-2.26.0.min.js'></script>\n" + div1_txt)
print("Saved: 1on1_dependency_sankey.html")

# ── CELL 7: Scatter plot (2-on-1 FDs) ───────────────────────
h = 30 * len(colsName)
if h > 1000:
    h = 1000

col1_list, col2_list, dep_list = [], [], []

for item in Uni_FD_set2:
    i, j = dic[item[0][0]], dic[item[0][1]]
    if i < j:
        col1_list.append(item[0][0])
        col2_list.append(item[0][1])
    else:
        col1_list.append(item[0][1])
        col2_list.append(item[0][0])
    dep_list.append(item[1])

fig_scatter = px.scatter(
    x=col1_list, y=col2_list, color=dep_list,
    height=h, title="COLUMN WISE DEPENDENCY"
)
fig_scatter.update_xaxes(categoryorder="array", categoryarray=colsName)
fig_scatter.update_yaxes(categoryorder="array", categoryarray=list(reversed(colsName)))
fig_scatter.update_layout(autosize=False, width=800, height=600)
fig_scatter.show()

div2_txt = plotly.offline.plot(fig_scatter, include_plotlyjs=False, output_type="div")
with open("2on1_dependency_scatter.html", "w") as f:
    f.write("<script src='https://cdn.plot.ly/plotly-2.26.0.min.js'></script>\n" + div2_txt)
print("Saved: 2on1_dependency_scatter.html")

# ── CELL 8: Save JSON results ────────────────────────────────
finaljson = {
    "keys": Uni_K,
    "equivalences": Uni_E,
    "functional_dependencies": Uni_FD,
}
with open("fd_results.json", "w") as f:
    f.write(json.dumps(finaljson, indent=2))
print("Saved: fd_results.json")
print(json.dumps(finaljson, indent=2))

Shape: (709, 9)
  StudentID    Major    Year CourseID  CourseName Credits Department  \
0      S001  Physics  Senior      C04  Statistics       3       Math   
1      S001  Physics  Senior      C01  Algorithms       4         CS   
2      S002       CS  Senior      C01  Algorithms       4         CS   
3      S002       CS  Senior      C07          OS       3         CS   
4      S002       CS  Senior      C04  Statistics       3       Math   

       Building Grade  
0  Science Bldg     A  
1     Tech Hall     B  
2     Tech Hall     A  
3     Tech Hall     F  
4  Science Bldg     C  
Functional Dependencies:
  {StudentID} -> {Major}
  {StudentID} -> {Year}
  {CourseID} -> {CourseName}
  {CourseID} -> {Credits}
  {CourseID} -> {Department}
  {CourseID} -> {Building}
  {CourseName} -> {CourseID}
  {CourseName} -> {Credits}
  {CourseName} -> {Department}
  {CourseName} -> {Building}
  {Department} -> {Building}

Equivalences:
  {CourseID} <-> {CourseName}

Keys:
  {StudentID, CourseID}


Saved: 1on1_dependency_sankey.html


Saved: 2on1_dependency_scatter.html
Saved: fd_results.json
{
  "keys": [
    "{StudentID, CourseID}",
    "{StudentID, CourseName}"
  ],
  "equivalences": [
    "{CourseID} <-> {CourseName}"
  ],
  "functional_dependencies": [
    "{StudentID} -> {Major}",
    "{StudentID} -> {Year}",
    "{CourseID} -> {CourseName}",
    "{CourseID} -> {Credits}",
    "{CourseID} -> {Department}",
    "{CourseID} -> {Building}",
    "{CourseName} -> {CourseID}",
    "{CourseName} -> {Credits}",
    "{CourseName} -> {Department}",
    "{CourseName} -> {Building}",
    "{Department} -> {Building}",
    "{StudentID, Major} -> {Year}",
    "{StudentID, Year} -> {Major}",
    "{StudentID, CourseID} -> {Major}",
    "{StudentID, CourseID} -> {Year}",
    "{StudentID, CourseID} -> {CourseName}",
    "{StudentID, CourseID} -> {Credits}",
    "{StudentID, CourseID} -> {Department}",
    "{StudentID, CourseID} -> {Building}",
    "{StudentID, CourseID} -> {Grade}",
    "{StudentID, CourseName} -> {Major}",
    